# Hafta 3 — Keşifsel Veri Analizi

Hesaplamalar `src/eda.py`, grafikler `src/eda_plots.py` içindedir. Rapordaki sayıların kaynağı `notebooks/build_kesifsel_analiz.py` ile üretilen `hafta3_ozet.json` dosyasıdır.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
from src.config import CATEGORICAL_FEATURES, NOT_PREVIOUSLY_CONTACTED, NUMERIC_FEATURES, TARGET
from src.data_loader import load_raw
from src.eda import POSITIVE_LABEL, acceptance_rate_by, chi_square_tests, iqr_outlier_summary, mann_whitney_tests, numeric_summary
from src.eda_plots import plot_acceptance_rates, plot_correlation_heatmap

df = load_raw()
is_positive = df[TARGET] == POSITIVE_LABEL
overall_rate = is_positive.mean()
print(df.shape, f'genel kabul oranı: {overall_rate:.4f}')

## 1. Kategorik değişkenlere göre kabul oranları

In [ ]:
for column in ['job', 'month', 'poutcome', 'contact', 'education']:
    display(acceptance_rate_by(df, column).round(4))

In [ ]:
titles = {'poutcome': 'Önceki kampanya sonucu', 'contact': 'İletişim türü'}
plot_acceptance_rates({c: acceptance_rate_by(df, c) for c in titles}, overall_rate, titles);

## 2. Sayısal değişkenler: dağılım ve IQR aykırı değerleri

In [ ]:
display(numeric_summary(df, NUMERIC_FEATURES).round(2))
display(iqr_outlier_summary(df, ['age', 'balance', 'campaign', 'previous']).round(2))
display(iqr_outlier_summary(df[df['pdays'] != NOT_PREVIOUSLY_CONTACTED], ['pdays', 'previous']).round(2))

## 3. Spearman korelasyonu

In [ ]:
correlation = df[NUMERIC_FEATURES].assign(**{TARGET: is_positive.astype(int)}).corr(method='spearman')
plot_correlation_heatmap(correlation, 'Spearman korelasyonu');

## 4. İstatistiksel testler

In [ ]:
display(chi_square_tests(df, CATEGORICAL_FEATURES).round(4))
display(mann_whitney_tests(df, NUMERIC_FEATURES).round(4))